In [1]:
# Cell 1 — Setup paths and folders
from pathlib import Path
import os
import re
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\barath\recomart-pipeline")
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

LOG_DIR = PROJECT_ROOT / "logs" / "orchestration"
MONITOR_DIR = PROJECT_ROOT / "logs" / "monitoring"
RUN_DIR = PROJECT_ROOT / "runs" / "prefect" / RUN_TS
EXEC_NOTEBOOK_DIR = RUN_DIR / "executed_notebooks"

for p in [LOG_DIR, MONITOR_DIR, RUN_DIR, EXEC_NOTEBOOK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_TS:", RUN_TS)
print("Working directory:", Path.cwd())


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
RUN_TS: 20260429_164131
Working directory: C:\Users\barath\recomart-pipeline


In [2]:
# Cell 2 — Install Prefect and dependencies

import sys
import subprocess
import importlib
import os

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        print(f"[OK] {pip_name} already available")
    except ImportError:
        print(f"[INSTALL] {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

packages = [
    ("prefect", "prefect"),
    ("papermill", "papermill"),
    ("nbformat", "nbformat"),
    ("nbclient", "nbclient"),
    ("ipykernel", "ipykernel"),
    ("requests", "requests"),
]

for import_name, pip_name in packages:
    ensure_package(import_name, pip_name)

os.environ["PREFECT_LOGGING_LEVEL"] = "INFO"
os.environ["PYTHONIOENCODING"] = "utf-8"

print("Dependency check complete.")



[OK] prefect already available
[OK] papermill already available
[OK] nbformat already available
[OK] nbclient already available
[OK] ipykernel already available
[OK] requests already available
Dependency check complete.


In [3]:
#Cell 3 — Start local Prefect server/UI
# Code Generated by Sidekick is for learning and experimentation purposes only.
import socket
import time
import os
import subprocess
import sys
import webbrowser
import requests

PREFECT_UI_URL = "http://127.0.0.1:4200"
PREFECT_API_URL = "http://127.0.0.1:4200/api"
PREFECT_SERVER_LOG = LOG_DIR / f"prefect_server_{RUN_TS}.log"

def is_port_open(host="127.0.0.1", port=4200):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

def wait_for_prefect_api(api_url=PREFECT_API_URL, timeout=120):
    start = time.time()
    health_url = f"{api_url}/health"
    while time.time() - start < timeout:
        try:
            response = requests.get(health_url, timeout=3)
            if response.status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False

server_process = None

if is_port_open():
    print("Prefect server already running on port 4200.")
else:
    log_file = open(PREFECT_SERVER_LOG, "w", encoding="utf-8")
    server_process = subprocess.Popen(
        [sys.executable, "-m", "prefect", "server", "start"],
        cwd=str(PROJECT_ROOT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env={**os.environ, "PREFECT_API_URL": PREFECT_API_URL}
    )
    print("Started Prefect server process:", server_process.pid)
    print("Server log:", PREFECT_SERVER_LOG)

ready = wait_for_prefect_api()
if not ready:
    raise RuntimeError("Prefect server did not become ready within timeout.")

os.environ["PREFECT_API_URL"] = PREFECT_API_URL
print("Prefect UI:", PREFECT_UI_URL)
print("Prefect API:", PREFECT_API_URL)

try:
    webbrowser.open(PREFECT_UI_URL)
except Exception:
    pass


Started Prefect server process: 10756
Server log: C:\Users\barath\recomart-pipeline\logs\orchestration\prefect_server_20260429_164131.log
Prefect UI: http://127.0.0.1:4200
Prefect API: http://127.0.0.1:4200/api


In [4]:
# Cell 4 — Define your notebook paths
NOTEBOOK_PATHS = {
    "ingest_dummyjson": PROJECT_ROOT / "src" / "01-ingestion" / "dummyjson_ingestion.ipynb",
    "ingest_retailrocket": PROJECT_ROOT / "src" / "01-ingestion" / "retailrocket_ingestion.ipynb",
    "validate_raw": PROJECT_ROOT / "src" / "02-validation" / "run_validation.ipynb",
    "eda_prep": PROJECT_ROOT / "src" / "03-eda-prep" / "eda_prep-copy.ipynb",
    "materialize_features": PROJECT_ROOT / "src" / "04-features" / "materialize_features.ipynb",
    "registry_loader": PROJECT_ROOT / "src" / "04-features" / "registry_loader.ipynb",
    "train_item_based": PROJECT_ROOT / "src" / "05-training" / "item-based-collaborative.ipynb",
    "train_svd": PROJECT_ROOT / "src" / "05-training" / "svd.ipynb",
    "train_content_based": PROJECT_ROOT / "src" / "05-training" / "content-based-training.ipynb",
    "compare_models": PROJECT_ROOT / "src" / "05-training" / "comparision.ipynb",
}

for name, path in NOTEBOOK_PATHS.items():
    print(f"{name:22} -> {'FOUND' if path.exists() else 'MISSING'} | {path}")


ingest_dummyjson       -> FOUND | C:\Users\barath\recomart-pipeline\src\01-ingestion\dummyjson_ingestion.ipynb
ingest_retailrocket    -> FOUND | C:\Users\barath\recomart-pipeline\src\01-ingestion\retailrocket_ingestion.ipynb
validate_raw           -> FOUND | C:\Users\barath\recomart-pipeline\src\02-validation\run_validation.ipynb
eda_prep               -> FOUND | C:\Users\barath\recomart-pipeline\src\03-eda-prep\eda_prep-copy.ipynb
materialize_features   -> FOUND | C:\Users\barath\recomart-pipeline\src\04-features\materialize_features.ipynb
registry_loader        -> FOUND | C:\Users\barath\recomart-pipeline\src\04-features\registry_loader.ipynb
train_item_based       -> FOUND | C:\Users\barath\recomart-pipeline\src\05-training\item-based-collaborative.ipynb
train_svd              -> FOUND | C:\Users\barath\recomart-pipeline\src\05-training\svd.ipynb
train_content_based    -> FOUND | C:\Users\barath\recomart-pipeline\src\05-training\content-based-training.ipynb
compare_models         ->

In [5]:
# Cell 5 — Logging and notebook execution helpers
import logging

def _safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()

def get_logger(name: str):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)

    if not logger.handlers:
        log_file = LOG_DIR / f"{_safe_name(name)}_{RUN_TS}.log"
        fh = logging.FileHandler(log_file, encoding="utf-8")
        sh = logging.StreamHandler(sys.stdout)

        formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
        fh.setFormatter(formatter)
        sh.setFormatter(formatter)

        logger.addHandler(fh)
        logger.addHandler(sh)

    return logger

def write_status(stage: str, status: str, notebook_path: Path = None, error: str = None, output_notebook: Path = None):
    payload = {
        "run_ts": RUN_TS,
        "stage": stage,
        "status": status,
        "notebook_path": str(notebook_path) if notebook_path else None,
        "output_notebook": str(output_notebook) if output_notebook else None,
        "error": error,
        "event_ts": datetime.now().isoformat()
    }
    out_path = MONITOR_DIR / f"{_safe_name(stage)}_{status}_{RUN_TS}.json"
    out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return out_path

def execute_notebook(stage: str, notebook_path: Path):
    logger = get_logger(stage)

    if not notebook_path.exists():
        write_status(stage, "failed", notebook_path=notebook_path, error="Notebook file not found")
        raise FileNotFoundError(f"Notebook not found: {notebook_path}")

    output_dir = EXEC_NOTEBOOK_DIR / _safe_name(stage)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_notebook = output_dir / f"{notebook_path.stem}_executed.ipynb"

    cmd = [
        sys.executable,
        "-m",
        "papermill",
        str(notebook_path),
        str(output_notebook),
    ]

    logger.info("Starting stage: %s", stage)
    logger.info("Command: %s", " ".join(cmd))
    write_status(stage, "started", notebook_path=notebook_path, output_notebook=output_notebook)

    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
        env={**os.environ, "PREFECT_API_URL": PREFECT_API_URL}
    )

    if result.stdout:
        logger.info("STDOUT:\n%s", result.stdout)

    if result.stderr:
        logger.warning("STDERR:\n%s", result.stderr)

    if result.returncode != 0:
        write_status(
            stage,
            "failed",
            notebook_path=notebook_path,
            output_notebook=output_notebook,
            error=result.stderr[-3000:] if result.stderr else f"Return code {result.returncode}"
        )
        raise RuntimeError(f"{stage} failed with return code {result.returncode}")

    write_status(stage, "success", notebook_path=notebook_path, output_notebook=output_notebook)
    logger.info("Completed stage: %s", stage)
    return str(output_notebook)


In [6]:
# Cell 6 — Print DAG graph
from IPython.display import HTML, display

dag_html = """
<div style="font-family: Arial, sans-serif; line-height: 1.6;">
  <h3>RecoMart Prefect Orchestration DAG</h3>

  <div style="margin: 12px 0;">
    <span style="display:inline-block;padding:10px 14px;background:#86BC25;color:white;border-radius:8px;">DummyJSON Ingestion</span>
    <span style="font-size:22px;margin:0 8px;">→</span>
    <span style="display:inline-block;padding:10px 14px;background:#0076A8;color:white;border-radius:8px;">Validation</span>
  </div>

  <div style="margin: 12px 0;">
    <span style="display:inline-block;padding:10px 14px;background:#43B02A;color:white;border-radius:8px;">RetailRocket Ingestion</span>
    <span style="font-size:22px;margin:0 8px;">→</span>
    <span style="display:inline-block;padding:10px 14px;background:#0076A8;color:white;border-radius:8px;">Validation</span>
  </div>

  <div style="margin: 12px 0;">
    <span style="display:inline-block;padding:10px 14px;background:#0076A8;color:white;border-radius:8px;">Validation</span>
    <span style="font-size:22px;margin:0 8px;">→</span>
    <span style="display:inline-block;padding:10px 14px;background:#1D4F91;color:white;border-radius:8px;">EDA / Preparation</span>
    <span style="font-size:22px;margin:0 8px;">→</span>
    <span style="display:inline-block;padding:10px 14px;background:#005587;color:white;border-radius:8px;">Materialize Features</span>
    <span style="font-size:22px;margin:0 8px;">→</span>
    <span style="display:inline-block;padding:10px 14px;background:#046E38;color:white;border-radius:8px;">Registry Loader</span>
  </div>

  <div style="margin: 12px 0;">
    <span style="display:inline-block;padding:10px 14px;background:#007CB0;color:white;border-radius:8px;">Item-Based Training</span>
    <span style="font-size:22px;margin:0 8px;">↘</span>
    <span style="display:inline-block;padding:10px 14px;background:#62B5E5;color:white;border-radius:8px;">Content-Based Training</span>
  </div>

  <div style="margin: 12px 0;">
    <span style="display:inline-block;padding:10px 14px;background:#00A9E0;color:white;border-radius:8px;">SVD Training</span>
    <span style="font-size:22px;margin:0 8px;">↗</span>
    <span style="display:inline-block;padding:10px 14px;background:#62B5E5;color:white;border-radius:8px;">Content-Based Training</span>
    <span style="font-size:22px;margin:0 8px;">→</span>
    <span style="display:inline-block;padding:10px 14px;background:#63666A;color:white;border-radius:8px;">Comparison</span>
  </div>
</div>
"""

display(HTML(dag_html))


In [7]:
# Cell 7 — Define Prefect tasks and flow
from prefect import flow, task, get_run_logger

@task(name="ingest_dummyjson", retries=2, retry_delay_seconds=20)
def ingest_dummyjson():
    return execute_notebook("ingest_dummyjson", NOTEBOOK_PATHS["ingest_dummyjson"])

@task(name="ingest_retailrocket", retries=2, retry_delay_seconds=20)
def ingest_retailrocket():
    return execute_notebook("ingest_retailrocket", NOTEBOOK_PATHS["ingest_retailrocket"])

@task(name="validate_raw", retries=1, retry_delay_seconds=10)
def validate_raw():
    return execute_notebook("validate_raw", NOTEBOOK_PATHS["validate_raw"])

@task(name="eda_prep", retries=1, retry_delay_seconds=10)
def eda_prep():
    return execute_notebook("eda_prep", NOTEBOOK_PATHS["eda_prep"])

@task(name="materialize_features", retries=1, retry_delay_seconds=10)
def materialize_features():
    return execute_notebook("materialize_features", NOTEBOOK_PATHS["materialize_features"])

@task(name="registry_loader", retries=1, retry_delay_seconds=10)
def registry_loader():
    return execute_notebook("registry_loader", NOTEBOOK_PATHS["registry_loader"])

@task(name="train_item_based", retries=0)
def train_item_based():
    return execute_notebook("train_item_based", NOTEBOOK_PATHS["train_item_based"])

@task(name="train_svd", retries=0)
def train_svd():
    return execute_notebook("train_svd", NOTEBOOK_PATHS["train_svd"])

@task(name="train_content_based", retries=0)
def train_content_based():
    return execute_notebook("train_content_based", NOTEBOOK_PATHS["train_content_based"])

@task(name="compare_models", retries=0)
def compare_models():
    return execute_notebook("compare_models", NOTEBOOK_PATHS["compare_models"])

@flow(name="recomart_assignment_prefect_flow", log_prints=True)
def recomart_assignment_prefect_flow():
    logger = get_run_logger()
    logger.info("Pipeline started")

    pipeline_status = {
        "run_ts": RUN_TS,
        "status": "started",
        "event_ts": datetime.now().isoformat()
    }
    (MONITOR_DIR / f"pipeline_started_{RUN_TS}.json").write_text(
        json.dumps(pipeline_status, indent=2), encoding="utf-8"
    )

    try:
        dummy_future = ingest_dummyjson.submit()
        retail_future = ingest_retailrocket.submit()

        dummy_future.result()
        retail_future.result()

        validate_raw.submit().result()
        eda_prep.submit().result()

        materialize_features.submit().result()
        registry_loader.submit().result()

        item_future = train_item_based.submit()
        svd_future = train_svd.submit()

        item_future.result()
        svd_future.result()

        train_content_based.submit().result()
        compare_models.submit().result()

        pipeline_status["status"] = "success"
        pipeline_status["event_ts"] = datetime.now().isoformat()
        (MONITOR_DIR / f"pipeline_success_{RUN_TS}.json").write_text(
            json.dumps(pipeline_status, indent=2), encoding="utf-8"
        )

        logger.info("Pipeline completed successfully")

    except Exception as exc:
        pipeline_status["status"] = "failed"
        pipeline_status["error"] = str(exc)
        pipeline_status["event_ts"] = datetime.now().isoformat()
        (MONITOR_DIR / f"pipeline_failed_{RUN_TS}.json").write_text(
            json.dumps(pipeline_status, indent=2), encoding="utf-8"
        )
        logger.exception("Pipeline failed: %s", exc)
        raise


In [8]:
# Cell 8 — Validate all notebook files exist
missing = [str(path) for path in NOTEBOOK_PATHS.values() if not path.exists()]

if missing:
    print("Missing notebook files:")
    for m in missing:
        print("-", m)
    raise FileNotFoundError("Fix missing notebook paths before running the flow.")
else:
    print("All notebook paths found.")


All notebook paths found.


In [9]:
#Cell 9 — Run the full orchestration flow
recomart_assignment_prefect_flow()


16:42:01.809 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8043
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

16:42:14.167 | INFO    | Flow run 'able-hare' - Beginning flow run 'able-hare' for flow 'recomart_assignment_prefect_flow'

16:42:14.179 | INFO    | Flow run 'able-hare' - Pipeline started

2026-04-29 16:42:14,260 | INFO | ingest_dummyjson | Starting stage: ingest_dummyjson
2026-04-29 16:42:14,266 | INFO | ingest_retailrocket | Starting stage: ingest_retailrocket


16:42:14.260 | INFO    | ingest_dummyjson - Starting stage: ingest_dummyjson

...

2026-04-29 17:07:14,863 | INFO | compare_models | Completed stage: compare_models


17:07:14.863 | INFO    | compare_models - Completed stage: compare_models

17:07:14.870 | INFO    | Task run 'compare_models-25f' - Finished in state Completed()

17:07:14.879 | INFO    | Flow run 'able-hare' - Pipeline completed successfully

17:07:15.974 | INFO    | Flow run 'able-hare' - Finished in state Completed()

In [10]:
# Cell 10 — Show logs, monitoring, and executed notebooks
print("Monitoring files:")
for p in sorted(MONITOR_DIR.glob(f"*{RUN_TS}*.json")):
    print("-", p)

print("\nExecuted notebooks:")
for p in sorted(EXEC_NOTEBOOK_DIR.rglob("*.ipynb")):
    print("-", p)

print("\nTask logs:")
for p in sorted(LOG_DIR.glob(f"*{RUN_TS}.log")):
    print("-", p)


Monitoring files:
- C:\Users\barath\recomart-pipeline\logs\monitoring\compare_models_started_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\compare_models_success_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\eda_prep_started_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\eda_prep_success_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\ingest_dummyjson_started_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\ingest_dummyjson_success_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\ingest_retailrocket_started_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\ingest_retailrocket_success_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\materialize_features_started_20260429_164131.json
- C:\Users\barath\recomart-pipeline\logs\monitoring\materialize_features_success_20260429_164131.json
- C:\Use

In [11]:
# Cell 11 — Optional cleanup: stop Prefect server started by notebook
if "server_process" in globals() and server_process is not None:
    server_process.terminate()
    print(f"Stopped Prefect server process: {server_process.pid}")
else:
    print("No notebook-started Prefect server process found.")


Stopped Prefect server process: 10756
